In [1]:
import numpy as np

In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [3]:
def log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

In [4]:
def leaf_weight(G, H, lam):
    return -G/ (H+lam)

In [5]:
class LeafNode:
    def __init__(self, w):
        self.w = w

class SplitNode:
    def __init__(self, feature, threshold, left, right):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right

In [6]:
def predict_tree(node, X):
    out = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        cur = node
        while isinstance(cur, SplitNode):
            cur = cur.left if X[i, cur.feature] <= cur.threshold else cur.right
        out[i] = cur.w
    return out


In [7]:
def xgboost_predict(H0, trees, eta, X):
    z = np.full(X.shape[0], H0)
    for tree in trees:
        z = z + eta * predict_tree(tree, X)
    return sigmoid(z)

In [8]:
def build_tree(X, g, h, depth, max_depth, lam, gamma):
    """Greedly grow one regression tree using the XGBoost gain criterion"""
    G, H = g.sum(), h.sum()

    if depth == max_depth or len(g) < 2:
        return LeafNode(w=leaf_weight(G, H, lam))

    best_gain, best_j, best_s = -np.inf, None, None
    n, d = X.shape

    for j in range(d):
        order = np.argsort(X[:, j]) # returns the index of orders of j-th column
        print(f"Order in {j}th round is: {order} \n") # eg. X[:, j] = [7, 2, 5, 1] -> order = [3, 1, 2, 0]

        Xj_sorted = X[order, j] # sort j-th column w.r.t "order"(holds index)
        g_sorted, h_sorted = g[order], h[order] # sort grad and hess exactly like X w.r.t "order"

        G_l = np.cumsum(g_sorted)   # calculates the gradient of all possible split of feat j (only left)
        H_l = np.cumsum(h_sorted)   # calculates the hessian of all possible split of feat j (only left)
        Gr = G - G_l    # gradient of right part
        Hr = H - H_l    # gradient of left part

        gain  = 0.5 * (G_l**2/(H_l+lam) + Gr**2/(Hr+lam) - G**2/(H+lam)) - gamma

        i_star = np.argmax(gain[:-1])
        if gain[i_star] > best_gain:
            best_gain = gain[i_star]
            best_j = j
            best_s = (Xj_sorted[i_star] + Xj_sorted[i_star+1])/2

    if best_gain <= 0:
        return LeafNode(w=leaf_weight(G, H, lam))

    left_mask = X[:, best_j] < best_s
    right_mask = ~left_mask

    left_child  = build_tree(X[left_mask],  g[left_mask],  h[left_mask],  depth+1, max_depth, lam, gamma)
    right_child = build_tree(X[right_mask], g[right_mask], h[right_mask], depth+1, max_depth, lam, gamma)

    return SplitNode(feature=best_j, threshold=best_s, left=left_child, right=right_child)


In [9]:
def xgboost_classifier_fit(X, y, T, eta, lam, gamma, max_depth):
    """
    X: (n, d) feature matrix
    y: (n,) binary labels in {0, 1}
    T: number of boosting rounds
    eta: learning rate
    lam: L2 regularization on leaf weights (lambda)
    gamma: min gain required to make a split (also per-leaf penalty)
    max_depth: max depth of each tree
    """

    history = {"loss": [], "grad_sum": [], "grad_absmean": [], "hess_mean": []}

    n = X.shape[0]
    p_bar = y.mean()
    H0 = np.log(p_bar/(1 - p_bar))
    z = np.full(n, H0)
    trees = []

    for m in range(T):
        p = sigmoid(z)
        g = p - y
        h = p * (1-p)

        r = -g                                    # pseudo-residual, y - p
        print(f"Round {m}: sum of residuals = {r.sum():.4f}") 

        history["loss"].append(log_loss(y, p))          # needs a log_loss(y, p) helper
        history["grad_sum"].append(g.sum())
        history["grad_absmean"].append(np.abs(g).mean())
        history["hess_mean"].append(h.mean())


        tree = build_tree(X, g, h, depth=0, max_depth=max_depth, lam=lam, gamma=gamma)
        trees.append(tree)

        print(len(trees))

        f_m = predict_tree(tree, X)
        z = z + eta * f_m

    import matplotlib.pyplot as plt
    plt.plot(history["loss"])
    plt.xlabel("Boosting round"); plt.ylabel("Log-loss"); plt.show()

    return H0, trees, eta


In [10]:
# import numpy as np
# from sklearn.model_selection import train_test_split

# np.random.seed(42)

# def make_dataset(n=200, d=2):
#     """
#     Two overlapping Gaussian blobs -> a binary classification problem
#     that isn't perfectly separable (so boosting actually has work to do).
#     """
#     n0 = n // 2
#     n1 = n - n0
#     n2 = n - n1

#     # Class 0: centered near (-1, -1)
#     X0 = np.random.randn(n0, d) * 1.2 + np.array([-1, -1])
#     y0 = np.zeros(n0)

#     # Class 1: centered near (2, 2)
#     X1 = np.random.randn(n1, d) * 1.2 + np.array([2, 2])
#     y1 = np.ones(n1)

#     # # Class 1: centered near (2, 2)
#     # X2 = np.random.randn(n2, d) * 1.2 + np.array([5, 5, 5, 5])
#     # y2 = np.ones(n2)

#     X = np.vstack([X0, X1])
#     y = np.concatenate([y0, y1])

#     # shuffle
#     idx = np.random.permutation(n)
#     return X[idx], y[idx]

# X, y = make_dataset(n=10, d=2)

# # print(X[0][1])

# # simple train/test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(X_train.shape, y_train.shape)   # (160, 2) (160,)
# print(X_test.shape,  y_test.shape)    # (40, 2) (40,)
# print("class balance (train):", y_train.mean())

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)

def make_dataset(n=200):

    n0 = n // 2
    n1 = n - n0

    # Class 0
    X0 = np.column_stack([
        np.random.choice(["small", "medium", "large"], n0),
        np.random.choice(["red", "blue", "green"], n0)
    ])
    y0 = np.zeros(n0)

    # Class 1
    X1 = np.column_stack([
        np.random.choice(["small", "medium", "large"], n1),
        np.random.choice(["red", "blue", "green"], n1)
    ])
    y1 = np.ones(n1)

    X = np.vstack([X0, X1])
    y = np.concatenate([y0, y1])

    idx = np.random.permutation(n)

    return X[idx], y[idx]


X, y = make_dataset(n=10)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, y_train.shape)
print(X_train)
print(y_train)

(8, 2) (8,)
[['large' 'red']
 ['large' 'blue']
 ['large' 'green']
 ['large' 'blue']
 ['small' 'green']
 ['medium' 'blue']
 ['small' 'blue']
 ['small' 'green']]
[1. 1. 0. 1. 0. 1. 1. 0.]


In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

np.random.seed(42)


def make_dataset(n=200):
    n0 = n // 2
    n1 = n - n0

    # Class 0
    X0 = np.column_stack([
        np.random.choice(["small", "medium", "large"], n0),
        np.random.choice(["red", "blue", "green"], n0)
    ])
    y0 = np.zeros(n0)

    # Class 1
    X1 = np.column_stack([
        np.random.choice(["small", "medium", "large"], n1),
        np.random.choice(["red", "blue", "green"], n1)
    ])
    y1 = np.ones(n1)

    X = np.vstack([X0, X1])
    y = np.concatenate([y0, y1])

    # Shuffle
    idx = np.random.permutation(n)

    return X[idx], y[idx]


# --------------------------------------------------
# 1. Create categorical dataset
# --------------------------------------------------

X, y = make_dataset(n=200)

print("Original X:")
print(X[:5])

print("\nOriginal shape:")
print(X.shape)


# --------------------------------------------------
# 2. Train/test split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# --------------------------------------------------
# 3. One-hot encode categorical features
# --------------------------------------------------

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)


# --------------------------------------------------
# 4. Model-ready data
# --------------------------------------------------

print("\nEncoded X_train:")
print(X_train_encoded[:5])

print("\nEncoded shape:")
print(X_train_encoded.shape)

print("\nFeature names:")
print(encoder.get_feature_names_out())

Original X:
[['medium' 'blue']
 ['large' 'green']
 ['medium' 'red']
 ['small' 'red']
 ['small' 'blue']]

Original shape:
(200, 2)

Encoded X_train:
[[0. 1. 0. 0. 0. 1.]
 [0. 1. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0.]]

Encoded shape:
(160, 6)

Feature names:
['x0_large' 'x0_medium' 'x0_small' 'x1_blue' 'x1_green' 'x1_red']


In [13]:
H0, trees, eta= xgboost_classifier_fit(
    X_train, y_train,
    T=20, eta=0.3, lam=1.0, gamma=0.0, max_depth=4
)

p_test = xgboost_predict(H0, trees, eta, X_test)
y_pred = (p_test > 0.5).astype(int)

accuracy = (y_pred == y_test).mean()
print("test accuracy:", accuracy)

Round 0: sum of residuals = 0.0000
Order in 0th round is: [116  36  37  40 115  43 112 103 102 101  50 100  98  91 119  59  88  64
  65  87  67  68  86  71  72  73  74  75  84  62  33  81 152  20 147 133
 143  16  13  24   9 131   4 154   2 129 127 156 124 140  19 137 120 145
  85 150 151  83 146  89  95 139 122 118 117  82 130 114 132  90 111 135
 138  97  96 121  93  92 104   0  79 158  42  39  38  34  27  26  25  21
  18  17  14  10   8   7   6  47  48  32  57  70  66  63  60  56  55  54
 159  53 109 136  52  15  69 141 142  12 144  11  61  49 148 149  77  78
   5 153   3 155   1 157  76 108 134  23 110  45  44 113 107  41 106  94
  35 105  22  80  30 123  29 125 126  28 128  99  58  46  31  51] 



TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [ ]:
def print_tree(node, depth=0, prefix="Root"):
    indent = "  " * depth
    if isinstance(node, LeafNode):
        print(f"{indent}{prefix} -> Leaf: w = {node.w:.4f}")
    else:  # SplitNode
        print(f"{indent}{prefix} -> Split: feature[{node.feature}] <= {node.threshold:.4f}")
        print_tree(node.left,  depth+1, prefix="Left ")
        print_tree(node.right, depth+1, prefix="Right")

In [ ]:
for m, tree in enumerate(trees):
    print(f"--- Tree {m} ---")
    print_tree(tree)
    print()

--- Tree 0 ---
Root -> Split: feature[0] <= -0.1463
  Left  -> Leaf: w = -0.7742
  Right -> Split: feature[1] <= 0.1131
    Left  -> Leaf: w = -0.3038
    Right -> Leaf: w = 1.1009

--- Tree 1 ---
Root -> Split: feature[1] <= 0.1131
  Left  -> Leaf: w = -0.7011
  Right -> Split: feature[0] <= -0.1463
    Left  -> Leaf: w = -0.2645
    Right -> Leaf: w = 0.9376

--- Tree 2 ---
Root -> Split: feature[1] <= 0.1131
  Left  -> Leaf: w = -0.6291
  Right -> Split: feature[0] <= -0.1463
    Left  -> Leaf: w = -0.2518
    Right -> Leaf: w = 0.8149

--- Tree 3 ---
Root -> Split: feature[0] <= -0.1463
  Left  -> Leaf: w = -0.5782
  Right -> Split: feature[1] <= 0.1131
    Left  -> Leaf: w = -0.2246
    Right -> Leaf: w = 0.7197

--- Tree 4 ---
Root -> Split: feature[1] <= 0.1131
  Left  -> Leaf: w = -0.5266
  Right -> Split: feature[0] <= -0.1463
    Left  -> Leaf: w = -0.2144
    Right -> Leaf: w = 0.6434

--- Tree 5 ---
Root -> Split: feature[0] <= -0.1463
  Left  -> Leaf: w = -0.4895
  Right -